# Color and Perception
## From light to an interactive Streamlit explanation

A colour is not stored in a wavelength, an object, an eye or a hexadecimal code. This lab follows the translations between them:

> **light source → spectrum → surface → spectrum reaching the eye → cone responses → neural processing → appearance → colour coordinates → display signal → new light**

We will build each link, identify what it preserves, and identify what it discards. The final section turns the model into a small Streamlit app.


## What you will be able to explain

By the end of the lab you should be able to:

1. distinguish a wavelength from a spectral power distribution;
2. explain why three cone classes compress a spectrum to three responses;
3. explain metamerism without claiming that a display reproduces the original light;
4. construct the CIE 1931 spectral locus from the official colour-matching table;
5. interpret a gamut triangle and the line of purples; and
6. turn a Python calculation into a Streamlit app with widgets, charts and explanatory text.


## Setup

Install the packages once from a terminal:

```bash
python -m pip install numpy pandas matplotlib streamlit jupyter
```

Keep this notebook inside its supplied folder. The official CIE table is stored at `data/CIE_xyz_1931_2deg.csv`; no internet connection is needed during the lab.


In [ ]:
from pathlib import Path
import importlib.util

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.figsize": (9, 4.6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})

DATA_FILE = Path("data/CIE_xyz_1931_2deg.csv")
print("CIE table:", "found" if DATA_FILE.exists() else "NOT FOUND")
print("Streamlit:", "installed" if importlib.util.find_spec("streamlit") else "install before the app section")


# 1 · What light is

Light is electromagnetic radiation. A wavelength describes one repeating spatial period of that radiation; wavelength is measured here in nanometres (nm). Human vision responds to only a limited interval of the electromagnetic spectrum, conventionally approximated as 380–780 nm.

A real light source usually emits many wavelengths at once. Its **spectral power distribution** (SPD) records how much power is present at each wavelength. It is a curve, not a single colour name:

\[
P(\lambda) = \text{power at wavelength }\lambda.
\]

Sunlight, a candle and a display pixel can look pale or white while having quite different SPDs. Calling the light “white” reports an appearance under particular conditions; it does not specify the physical recipe.


In [ ]:
def gaussian(x, centre, width):
    return np.exp(-0.5 * ((x - centre) / width) ** 2)

def planck(wavelength_nm, temperature):
    wavelength_m = wavelength_nm * 1e-9
    c2 = 1.438776877e-2
    values = 1 / (wavelength_m**5 * np.expm1(c2 / (wavelength_m * temperature)))
    return values / values.max()

wavelengths = np.arange(380, 781)
spectra = {
    "daylight-like source": planck(wavelengths, 5800),
    "candle-like source": planck(wavelengths, 1800),
    "LCD display (white-LED backlight)": (
        gaussian(wavelengths, 450, 9)
        + 0.58 * gaussian(wavelengths, 535, 30)
        + 0.65 * gaussian(wavelengths, 612, 25)
    ),
}
spectra["LCD display (white-LED backlight)"] /= spectra["LCD display (white-LED backlight)"].max()

fig, ax = plt.subplots()
for label, spectrum in spectra.items():
    ax.plot(wavelengths, spectrum, lw=2.5, label=label)
ax.set(xlabel="wavelength (nm)", ylabel="relative spectral power",
       xlim=(380, 780), title="Different physical recipes")
ax.legend()
plt.show()


## Light from an object is a product

An ordinary surface does not contain a fixed colour signal. The spectrum reaching the eye depends on both the illumination and the surface:

\[
P_{\text{eye}}(\lambda)
= P_{\text{illuminant}}(\lambda)\,R_{\text{surface}}(\lambda).
\]

`R_surface` is the fraction reflected at each wavelength. Change the lamp and the same surface sends a different spectrum to the eye. This is why colour management needs viewing conditions, not only object labels.


In [ ]:
illuminant = planck(wavelengths, 5200)
reflectance = 0.06 + 0.80 / (1 + np.exp(-(wavelengths - 585) / 13))
received = illuminant * reflectance

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True, sharey=True)
for ax, values, title in zip(
    axes,
    [illuminant, reflectance, received],
    ["illumination", "surface reflectance", "spectrum reaching the eye"],
):
    ax.plot(wavelengths, values, lw=2.5)
    ax.fill_between(wavelengths, values, alpha=.15)
    ax.set(title=title, xlabel="wavelength (nm)", xlim=(380, 780), ylim=(0, 1.05))
axes[0].set_ylabel("relative amount")
plt.show()


# 2 · What the eye does with light

Human daylight colour vision begins with three cone classes: S, M and L. Their labels mean **short-, middle- and long-wavelength sensitive**. They should not be renamed blue, green and red receptors: all three have broad, overlapping sensitivities, and two classes can respond strongly to the same wavelength.

For a teaching model, the response of cone class (i) is:

\[
q_i = \int P_{\text{eye}}(\lambda)s_i(\lambda)\,d\lambda,
\]

where (P_{\text{eye}}) is the incoming spectrum and (s_i) is that cone class's sensitivity curve. Multiply wavelength by wavelength, then add. One complete spectrum becomes three response values.

The “three buckets” analogy is useful if used carefully: each bucket has a different spectral filter, and its output is only a total. A single cone cannot report which wavelength caused its response. This is the **principle of univariance**.


In [ ]:
def cone_sensitivities(wavelengths):
    # Schematic teaching curves, not physiological reference data.
    return np.column_stack([
        gaussian(wavelengths, 445, 32),
        gaussian(wavelengths, 535, 43),
        gaussian(wavelengths, 565, 48),
    ])

cones = cone_sensitivities(wavelengths)
fig, ax = plt.subplots()
for label, curve, colour in zip(
    ["S", "M", "L"], cones.T, ["#5B4BCE", "#00A878", "#E44B35"]
):
    ax.plot(wavelengths, curve, lw=2.8, label=label, color=colour)
ax.set(xlabel="wavelength (nm)", ylabel="relative sensitivity",
       xlim=(380, 780), ylim=(0, 1.05),
       title="Schematic cone sensitivities overlap")
ax.legend()
plt.show()


In [ ]:
def cone_response(spectrum):
    return np.trapezoid(spectrum[:, None] * cones, wavelengths, axis=0)

response_rows = []
for label, spectrum in spectra.items():
    response = cone_response(spectrum)
    response_rows.append([label, *response])

responses = pd.DataFrame(response_rows, columns=["light", "S", "M", "L"])
responses[["S", "M", "L"]] = responses[["S", "M", "L"]].div(
    responses[["S", "M", "L"]].max(axis=1), axis=0
)
responses.round(3)


## What three cone responses explain—and what they do not

Three-channel sampling explains **colour matching**. If two spectra produce the same three responses for an observer under fixed conditions, the observer cannot distinguish them by colour in that matching task. Such spectra are **metamers**.

This is why a display can match the appearance of a spectral yellow using red and green primaries. The display does not recreate the original spectrum. It creates another spectrum that produces the required match.

Trichromacy does not explain everything about appearance. After the cones, the visual system compares signals across space and time, forms opponent channels, adapts to illumination and uses surrounding context. The same retinal input can therefore contribute to different appearances in different scenes.


# 3 · Metamerism: what a display exploits

The next calculation constructs two different, non-negative spectra with the same response under the CIE 1931 standard observer. The linear algebra is supplied; concentrate on the result.


In [ ]:
cie = pd.read_csv(
    DATA_FILE,
    header=None,
    names=["wavelength", "xbar", "ybar", "zbar"],
).query("380 <= wavelength <= 700").copy()

cie_wavelengths = cie["wavelength"].to_numpy()
cie_xyz = cie[["xbar", "ybar", "zbar"]].to_numpy()
response_matrix = cie_xyz.T

candidate = (
    np.sin(np.linspace(0, 10 * np.pi, len(cie)))
    + 0.35 * np.cos(np.linspace(0, 3 * np.pi, len(cie)))
)
invisible_component = candidate - response_matrix.T @ np.linalg.solve(
    response_matrix @ response_matrix.T,
    response_matrix @ candidate,
)
invisible_component /= np.max(np.abs(invisible_component))

spectrum_a = 0.55 + 0.28 * invisible_component
spectrum_b = 0.55 - 0.28 * invisible_component
XYZ_a = response_matrix @ spectrum_a
XYZ_b = response_matrix @ spectrum_b

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cie_wavelengths, spectrum_a, lw=2.5, label="spectrum A")
ax.plot(cie_wavelengths, spectrum_b, lw=2.5, label="spectrum B")
ax.set(xlabel="wavelength (nm)", ylabel="relative power",
       title="Different spectra")
ax.legend()
plt.show()

comparison = pd.DataFrame(
    [XYZ_a / XYZ_a.sum(), XYZ_b / XYZ_b.sum()],
    index=["A", "B"], columns=["X proportion", "Y proportion", "Z proportion"]
)
display(comparison.round(9))
print("Largest difference in unnormalised XYZ:", np.max(np.abs(XYZ_a - XYZ_b)))


# 4 · Context enters after the first three responses

Both centre patches below have exactly the same RGB values. If they appear different, the pixel did not change. The surround changed the visual system's comparison.

This matters for visualisation: a palette is never perceived as a list of isolated hexadecimal codes. Every colour is seen against neighbours, backgrounds and display conditions.


In [ ]:
image = np.ones((220, 620, 3))
image[:, :310] = 0.10
image[:, 310:] = 0.88
centre = np.array([0.38, 0.52, 0.68])
image[65:155, 100:210] = centre
image[65:155, 410:520] = centre

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(image)
ax.set_axis_off()
ax.set_title("Identical centre RGB: [0.38, 0.52, 0.68]")
plt.show()


# 5 · A short history of different questions

Colour theory did not advance by everybody answering one question more accurately. Different systems were built for different objects and tasks.

| Work | Object being studied | Question |
|---|---|---|
| Newton, *Opticks* (1704) | rays transformed by prisms | How does white light separate and recombine? |
| Goethe, *Theory of Colours* (1810) | colour as experienced | How do shadows, boundaries, afterimages and context affect appearance? |
| Maxwell, 1850s–60s | three-primary matches | How can one light be matched by mixtures of three others? |
| Munsell, *A Color Notation* (1905) | ordered surface samples | How can hue, value and chroma support practical comparison? |
| CIE, 1931 | average matching behaviour | How can laboratories and industries report compatible colour matches? |

Newton's optics and Goethe's observations of appearance operate at different points in the chain. Munsell is a system for ordering surface-colour samples. CIE XYZ is a mathematical system for recording standardised colour matches. Treating all four as rival versions of one colour wheel erases the problem each was designed to solve.


# 6 · From matching experiments to CIE XYZ

Colour measurement did not begin by placing sensors directly inside an eye. Observers adjusted three primary lights until one half of a small field matched a test light in the other half.

With the real primaries used in the 1931 RGB experiments, some test wavelengths could not be matched using positive amounts of all three primaries. The experimenter moved one primary to the test side. Algebra records that operation as a negative amount on the matching side.

The CIE then transformed those RGB matching results into the mathematical coordinates **X, Y and Z**. They are not three cone outputs and not three physical lamps. The transform was chosen so that:

- the standard colour-matching functions are non-negative;
- (Y) carries the standard photopic luminance quantity; and
- ordinary colour matching can be calculated with linear algebra.

For a spectrum (P(\lambda)):

\[
X=\int P(\lambda)\bar{x}(\lambda)d\lambda,\quad
Y=\int P(\lambda)\bar{y}(\lambda)d\lambda,\quad
Z=\int P(\lambda)\bar{z}(\lambda)d\lambda.
\]


In [ ]:
fig, ax = plt.subplots()
for column, label, colour in [
    ("xbar", "x̄", "#E44B35"),
    ("ybar", "ȳ", "#00A878"),
    ("zbar", "z̄", "#5B4BCE"),
]:
    ax.plot(cie["wavelength"], cie[column], lw=2.5, label=label, color=colour)
ax.set(xlabel="wavelength (nm)", ylabel="matching function",
       title="Official CIE 1931 2° colour-matching functions")
ax.legend()
plt.show()


# 7 · Removing overall scale: chromaticity

Multiplying a spectrum by two multiplies X, Y and Z by two. Its relative three-way balance is unchanged. Chromaticity coordinates keep that balance:

\[
x=\frac{X}{X+Y+Z},\qquad
y=\frac{Y}{X+Y+Z},\qquad
z=\frac{Z}{X+Y+Z}=1-x-y.
\]

Only two coordinates are needed because the three proportions sum to one. An xy point therefore omits the overall tristimulus scale; it is **not** a complete description of brightness or colour appearance.

To draw the spectral locus, take one monochromatic wavelength at a time. Read its \(\bar{x},\bar{y},\bar{z}\) values, divide by their sum and plot \((x,y)\). Repeating this over the visible interval traces the curved boundary.


In [ ]:
totals = cie_xyz.sum(axis=1)
spectral_xy = cie_xyz[:, :2] / totals[:, None]

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(spectral_xy[:, 0], spectral_xy[:, 1], color="black", lw=2.2,
        label="spectral locus")
ax.plot([spectral_xy[-1, 0], spectral_xy[0, 0]],
        [spectral_xy[-1, 1], spectral_xy[0, 1]],
        color="#9475CD", lw=2.2, label="line of purples")
for wavelength in [420, 460, 500, 540, 580, 620, 680]:
    row = np.argmin(np.abs(cie_wavelengths - wavelength))
    ax.annotate(str(wavelength), spectral_xy[row], xytext=(5, 4),
                textcoords="offset points", fontsize=8)
ax.set(xlim=(0, .8), ylim=(0, .9), xlabel="x", ylabel="y",
       title="CIE 1931 xy chromaticity diagram")
ax.set_aspect("equal", adjustable="box")
ax.legend()
plt.show()


## Why mixtures lie inside

Adding two lights adds their spectra. Because XYZ is linear, their XYZ values add too. After normalisation to xy, the mixture lies on the straight segment joining the two chromaticities. Its position depends on the relative amounts.

A general spectrum is a non-negative mixture of wavelength components, so its chromaticity lies in the convex region bounded by the spectral locus and the straight **line of purples**. Purple is on that closure because it mixes light from opposite ends of the visible spectrum; it is not a single spectral wavelength.


In [ ]:
blue = spectral_xy[np.argmin(np.abs(cie_wavelengths - 460))]
red = spectral_xy[np.argmin(np.abs(cie_wavelengths - 610))]
amounts = np.linspace(0, 1, 9)
mixtures = amounts[:, None] * blue + (1 - amounts[:, None]) * red

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(np.r_[spectral_xy[:, 0], spectral_xy[0, 0]],
        np.r_[spectral_xy[:, 1], spectral_xy[0, 1]], color="#999", lw=1.5)
ax.plot(mixtures[:, 0], mixtures[:, 1], color="black", lw=2)
ax.scatter(mixtures[:, 0], mixtures[:, 1], c=amounts,
           cmap="coolwarm", s=75, edgecolor="white")
ax.set(xlim=(0, .8), ylim=(0, .9), xlabel="x", ylabel="y",
       title="Mixtures fall on the line between two lights")
ax.set_aspect("equal", adjustable="box")
plt.show()


# 8 · A display gamut is a triangle, not the horseshoe

A display has three physical primaries. Positive mixtures of those primaries produce chromaticities inside the triangle joining their coordinates. A wider triangle reaches more chromaticities, but no three real primaries cover the entire spectral locus.

The diagram is not a screenshot of all visible colours. Your notebook itself is displayed through your monitor's gamut, so colours outside that gamut cannot be shown faithfully even if their coordinates can be drawn.


In [ ]:
GAMUTS = {
    "sRGB / Rec.709": np.array([[0.640, 0.330], [0.300, 0.600], [0.150, 0.060]]),
    "Display P3": np.array([[0.680, 0.320], [0.265, 0.690], [0.150, 0.060]]),
    "Rec.2020": np.array([[0.708, 0.292], [0.170, 0.797], [0.131, 0.046]]),
}

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(np.r_[spectral_xy[:, 0], spectral_xy[0, 0]],
        np.r_[spectral_xy[:, 1], spectral_xy[0, 1]], color="black", lw=2)
for name, triangle in GAMUTS.items():
    closed = np.vstack([triangle, triangle[0]])
    ax.plot(closed[:, 0], closed[:, 1], marker="o", lw=2, label=name)
ax.set(xlim=(0, .8), ylim=(0, .9), xlabel="x", ylabel="y",
       title="Three display primaries define a triangular gamut")
ax.set_aspect("equal", adjustable="box")
ax.legend()
plt.show()


# 9 · What this means for data visualisation

Colour is a useful but conditional encoding channel.

- **Hue** is effective for distinguishing a modest number of categories; it does not supply a natural numerical order.
- **Lightness** can carry order when it changes monotonically.
- **Chroma** changes emphasis but supports only coarse comparisons.
- Equal steps in RGB or HSL are not equal perceptual steps.
- A palette must be tested against its actual background, at its actual mark size, and under relevant colour-vision differences.
- Important distinctions need a redundant cue such as position, direct labelling, shape or line style.

The screen receives codes. The viewer experiences comparisons in context. Designing only the codes ignores the middle of the chain.


# 10 · Streamlit: turn the calculation into an explanatory interface

A notebook runs cells when you ask. A Streamlit app is an ordinary Python script that **reruns from top to bottom whenever a widget changes**.

The basic pattern is:

```text
widget produces a Python value
             ↓
ordinary Python transforms that value
             ↓
st.* commands draw the new result
```

Common translations:

| Notebook or script | Streamlit app |
|---|---|
| fixed value `wavelength = 550` | widget `wavelength = st.slider(...)` |
| `print(value)` | `st.write(value)` or `st.metric(...)` |
| `display(df)` | `st.dataframe(df)` or `st.table(df)` |
| `plt.show()` | `st.pyplot(fig)` |
| headings in Markdown cells | `st.title()`, `st.header()`, `st.markdown()` |

Start with the smallest possible app.


In [ ]:
from pathlib import Path

HELLO_APP = 'import streamlit as st\n\nst.title("My first colour app")\nwavelength = st.slider("Wavelength (nm)", 380, 700, 550)\nst.write("Selected wavelength:", wavelength, "nm")\nst.caption("Moving the slider reruns this script from top to bottom.")\n'
hello_path = Path("hello_colour.py")
hello_path.write_text(HELLO_APP, encoding="utf-8")
print(f"Wrote {hello_path.resolve()}")
print("Run: python -m streamlit run hello_colour.py")


Open a terminal in this notebook folder and run:

```bash
python -m streamlit run hello_colour.py
```

Streamlit opens a local page, normally at `http://localhost:8501`. Move the slider and watch the printed wavelength update. Nothing calls a special event handler: the new widget value is assigned and the whole script reruns.

Stop the server with `Ctrl+C` when finished.


## Build the complete spectrum explorer

The next cell writes a complete app. Read it in this order:

1. imports and page configuration;
2. functions that contain the model;
3. sidebar widgets that create the current input values;
4. calculations derived from those values; and
5. tabs that present three different representations.

The app uses `@st.cache_data` only for the fixed CIE table. The spectrum and plots must recompute when a widget changes.


In [ ]:
APP_SOURCE = 'from pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\n\nst.set_page_config(page_title="Color and Perception", page_icon="👁️", layout="wide")\n\nDATA_FILE = Path(__file__).parent / "data" / "CIE_xyz_1931_2deg.csv"\n\n\n@st.cache_data\ndef load_cie():\n    return pd.read_csv(\n        DATA_FILE,\n        header=None,\n        names=["wavelength", "xbar", "ybar", "zbar"],\n    ).query("380 <= wavelength <= 700")\n\n\ndef gaussian(x, centre, width):\n    return np.exp(-0.5 * ((x - centre) / width) ** 2)\n\n\ndef cone_sensitivities(wavelengths):\n    """Schematic curves: useful for the model, not physiological reference data."""\n    return np.column_stack([\n        gaussian(wavelengths, 445, 32),\n        gaussian(wavelengths, 535, 43),\n        gaussian(wavelengths, 565, 48),\n    ])\n\n\nst.title("Color and Perception")\nst.caption("Change a physical spectrum; inspect what survives each representation.")\n\nst.sidebar.header("Light source")\ncentre = st.sidebar.slider("Peak wavelength (nm)", 400, 680, 540, 1)\nwidth = st.sidebar.slider("Spectral width (nm)", 2, 100, 28, 1)\nsecond_peak = st.sidebar.checkbox("Add a second peak")\nsecond_centre = st.sidebar.slider("Second peak (nm)", 400, 680, 620, 1, disabled=not second_peak)\n\nwavelengths = np.arange(380, 701)\nspectrum = gaussian(wavelengths, centre, width)\nif second_peak:\n    spectrum = spectrum + 0.65 * gaussian(wavelengths, second_centre, width)\n\ncones = cone_sensitivities(wavelengths)\nresponses = np.trapezoid(spectrum[:, None] * cones, wavelengths, axis=0)\nresponses = responses / responses.max()\n\ncie = load_cie()\nsampled_spectrum = np.interp(cie["wavelength"], wavelengths, spectrum)\nXYZ = cie[["xbar", "ybar", "zbar"]].to_numpy().T @ sampled_spectrum\nxy = XYZ[:2] / XYZ.sum()\n\nworld_tab, eye_tab, map_tab = st.tabs(["1 · Spectrum", "2 · Eye", "3 · Chromaticity"])\n\nwith world_tab:\n    st.subheader("The physical input is a spectrum")\n    fig, ax = plt.subplots(figsize=(9, 3.8))\n    ax.plot(wavelengths, spectrum, color="#0F4BEB", lw=3)\n    ax.fill_between(wavelengths, spectrum, color="#00B7E0", alpha=.18)\n    ax.set(xlabel="wavelength (nm)", ylabel="relative power", xlim=(380, 700))\n    st.pyplot(fig, clear_figure=True)\n    st.info("The peak wavelength is one property of this light. The full curve is the input.")\n\nwith eye_tab:\n    st.subheader("The model compresses the spectrum to three responses")\n    columns = st.columns(3)\n    for column, label, value in zip(columns, ["S", "M", "L"], responses):\n        column.metric(f"{label}-cone response", f"{value:.3f}")\n    fig, ax = plt.subplots(figsize=(9, 3.8))\n    for label, curve, colour in zip(["S", "M", "L"], cones.T, ["#5B4BCE", "#00A878", "#E44B35"]):\n        ax.plot(wavelengths, curve, lw=2.5, label=label, color=colour)\n    ax.set(xlabel="wavelength (nm)", ylabel="schematic sensitivity", xlim=(380, 700), ylim=(0, 1.05))\n    ax.legend()\n    st.pyplot(fig, clear_figure=True)\n    st.warning("These are schematic cone curves. Colour appearance also depends on adaptation, context and later neural processing.")\n\nwith map_tab:\n    st.subheader("CIE xy keeps chromaticity and leaves out overall scale")\n    xyz = cie[["xbar", "ybar", "zbar"]].to_numpy()\n    locus = xyz[:, :2] / xyz.sum(axis=1)[:, None]\n    fig, ax = plt.subplots(figsize=(6.4, 5.4))\n    ax.plot(np.r_[locus[:, 0], locus[0, 0]], np.r_[locus[:, 1], locus[0, 1]], color="#111", lw=2)\n    ax.scatter([xy[0]], [xy[1]], s=130, color="#FF2305", edgecolor="white", linewidth=1.5, zorder=3)\n    ax.set(xlim=(0, .8), ylim=(0, .9), xlabel="x", ylabel="y")\n    ax.set_aspect("equal", adjustable="box")\n    st.pyplot(fig, clear_figure=True)\n    st.metric("Current chromaticity", f"x={xy[0]:.3f}, y={xy[1]:.3f}")\n    st.caption("An xy point is not a spectrum and not a complete colour appearance.")\n\nst.divider()\nst.subheader("Your extension")\nst.write("Add an intensity slider that multiplies the entire spectrum. Cone-response magnitudes should change; x and y should remain almost unchanged.")\n'
app_path = Path("vision_colour_app.py")
app_path.write_text(APP_SOURCE, encoding="utf-8")
compile(APP_SOURCE, str(app_path), "exec")
print(f"Wrote {app_path.resolve()} ({len(APP_SOURCE.splitlines())} lines)")
print("Run: python -m streamlit run vision_colour_app.py")


## Your extension: separate intensity from chromaticity

Add an `Intensity` slider to the sidebar and multiply the entire spectrum by its value.

**Prediction before coding:** Which outputs should change? Which should remain stable?

**Definition of done:**

- the spectrum height changes;
- the unnormalised S, M and L responses change;
- the chromaticity point remains in the same place, apart from rounding;
- the app includes one sentence explaining why.

This is a better test than “the app runs”: it checks whether the interface preserves the distinction between amount of light and its chromaticity.


In [ ]:
# Record your intended edit or paste the revised block here.
# Then make the same change in vision_colour_app.py and test it in the browser.

my_change = ""
print(my_change or "Describe your change before editing the app.")


## Review the app as a visual explanation

Exchange screens with another student. Ask them to answer without seeing the code:

1. What is the physical input?
2. Where is that input reduced to three numbers?
3. What does the xy point preserve and omit?
4. Which curves are official data and which are schematic?
5. What claim would be wrong to make from this app?

Revise one label, caption or arrangement based on where they hesitate.


# Sources

- CIE, [CIE 1931 2° colour-matching functions](https://cie.co.at/datatable/cie-1931-colour-matching-functions-2-degree-observer), DOI `10.25039/CIE.DS.xvudnb9b`.
- Kalloniatis and Luu, [Color Perception](https://www.webvision.pitt.edu/book/part-viii-psychophysics-of-vision/color-perception/), *Webvision*.
- Isaac Newton, [*Opticks* (1704)](https://www.newtonproject.ox.ac.uk/view/texts/normalized/NATP00034).
- J. W. von Goethe, [*Theory of Colours* (1810)](https://www.gutenberg.org/ebooks/50572).
- A. H. Munsell, [*A Color Notation* (1905)](https://www.gutenberg.org/ebooks/26054).
- W3C, [CSS Color Module Level 4](https://www.w3.org/TR/css-color-4/).
- Streamlit, [Basic concepts](https://docs.streamlit.io/get-started/fundamentals/main-concepts).

**Model boundaries:** the cone curves and wavelength-to-display approximations in this lesson are schematic. CIE data are standard-observer colour-matching functions, not measured cone sensitivities. The app does not model adaptation, spatial context, individual observer variation, fluorescence or device calibration.
